In [6]:
# --- 0. INSTALLAZIONE E IMPORT ---
import os
# Installa il motore per Excel (se manca lo installa, altrimenti prosegue)
os.system('pip install xlsxwriter')

import yfinance as yf
import pandas as pd
import numpy as np
import requests
import io
import time
import random
import xlsxwriter

# --- 1. FUNZIONE ROBUSTA PER LISTA AZIONI ---
def get_sp500_tickers_robust():
    """Recupera la lista azioni da fonti multiple in caso di errore"""
    tickers = []

    # METODO 1: GitHub Datasets
    print("Tentativo 1: Scaricamento da Database GitHub...")
    try:
        url = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv"
        s = requests.get(url).content
        df = pd.read_csv(io.StringIO(s.decode('utf-8')))
        tickers = df['Symbol'].tolist()
        print(f"-> Successo! Trovati {len(tickers)} ticker.")
        return tickers
    except Exception as e:
        print(f"-> Fallito ({e}). Passo al metodo 2...")

    # METODO 2: Fallback Manuale (Sicurezza)
    print("Tentativo 2: Caricamento lista di emergenza...")
    tickers = [
        "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "TSLA", "META", "BRK-B", "UNH", "JNJ",
        "XOM", "JPM", "PG", "V", "LLY", "HD", "MA", "CVX", "ABBV", "PEP",
        "KO", "MRK", "BAC", "COST", "PFE", "TMO", "AVGO", "CSCO", "MCD", "DIS"
    ]
    print(f"-> Caricata lista ridotta di {len(tickers)} azioni.")
    return tickers

# --- 2. INPUT UTENTE ---
print("--- SISTEMA DI SELEZIONE AZIONARIA (V6.0 Final) ---\n")

try:
    risk_free_rate = float(input("1. Inserisci il Tasso Risk Free USA (es. 4.2): "))
    risk_free_decimal = risk_free_rate / 100

    print("\n2. Trend Bilancio Banche Centrali (Liquidità)?")
    print("   [1] In Aumento (Scenario 'Most Liquid')")
    print("   [2] In Diminuzione (Scenario 'Least Liquid')")
    liquidity_input = input("   Digitare 1 o 2: ")
    is_liquidity_increasing = True if liquidity_input == '1' else False

    benchmark_ticker = input("\n3. Indice di Riferimento (Invio per S&P500): ").upper()
    if not benchmark_ticker: benchmark_ticker = "^GSPC"

except ValueError:
    print("Errore input. Riavviare.")
    exit()

# --- 3. LOGICA MACRO ---
tassi_alti = risk_free_rate > 3.0
scenario_type = ""

if not tassi_alti and is_liquidity_increasing:
    scenario_type = "AGGRESSIVE (Stock D)"
elif tassi_alti and not is_liquidity_increasing:
    scenario_type = "DEFENSIVE (Stock A)"
else:
    scenario_type = "NEUTRAL/QUALITY (Stock B/C)"
print(f"\nSCENARIO RILEVATO: {scenario_type}")

# --- 4. PREPARAZIONE DATI ---
print("\n--- AVVIO RECUPERO DATI ---")
tickers_list = get_sp500_tickers_robust()
tickers_list = [t.replace('.', '-') for t in tickers_list]
# Scommenta la riga sotto se vuoi fare un test veloce (prime 50 azioni)
# tickers_list = tickers_list[:50]
tickers_data = tickers_list + [benchmark_ticker]

# --- 5. FUNZIONI ANALISI ---
def get_rrg_status(stock_data, bench_data):
    if len(stock_data) < 100 or len(bench_data) < 100: return "N/A"
    rs = stock_data['Close'] / bench_data['Close']
    rs_ratio = 100 + ((rs - rs.rolling(window=50).mean()) / rs.std()) * 10
    rs_momentum = 100 + ((rs_ratio - rs_ratio.shift(10)))

    curr_r = rs_ratio.iloc[-1]
    curr_m = rs_momentum.iloc[-1]

    if curr_r > 100 and curr_m > 100: return "LEADING"
    if curr_r > 100 and curr_m < 100: return "WEAKENING"
    if curr_r < 100 and curr_m < 100: return "LAGGING"
    return "IMPROVING"

def check_scenario_match(dati, scenario):
    match = "NO"
    # Scenario Difensivo
    if scenario == "DEFENSIVE (Stock A)":
        if dati['D/E'] is not None and dati['D/E'] < 0.8:
            if dati['Beta'] is not None and dati['Beta'] < 1.0:
                match = "SI (Difensivo)"
    # Scenario Aggressivo
    elif scenario == "AGGRESSIVE (Stock D)":
        if dati['Beta'] is not None and dati['Beta'] > 1.1:
            match = "SI (Aggressivo)"
    # Scenario Quality
    else:
        if dati['ROE'] is not None and dati['ROE'] > 0.15:
             if dati['D/E'] is not None and dati['D/E'] < 1.0:
                match = "SI (Quality)"
    return match

def analizza_fondamentali(ticker_obj, risk_free, scenario):
    try:
        info = ticker_obj.info
    except:
        return None

    dati = {
        'Prezzo': info.get('currentPrice', 0),
        'P/E': info.get('trailingPE', -999),
        'D/E': info.get('debtToEquity', 999),
        'EPS': info.get('trailingEps', 0),
        'MOL_Margin': info.get('ebitdaMargins', 0),
        'Profit_Margin': info.get('profitMargins', 0),
        'ROE': info.get('returnOnEquity', 0),
        'Beta': info.get('beta', 1),
        'Settore': info.get('sector', 'N/A'),
        'Note': ""
    }

    # Fix D/E se arriva in percentuale
    if dati['D/E'] is not None and dati['D/E'] > 10:
        dati['D/E'] /= 100

    score = 0

    # --- BLOCCO DI CALCOLO PUNTEGGIO (Quello che dava errore) ---
    try:
        # 1. Debito
        if dati['D/E'] is not None and dati['D/E'] < 0.5: score += 1
        elif dati['D/E'] is not None and dati['D/E'] <= 1: score += 0.5
        else: dati['Note'] += "Debito Alto; "

        # 2. P/E
        if dati['P/E'] is not None and dati['P/E'] > 0: score += 1

        # 3. EPS
        if dati['EPS'] is not None and dati['EPS'] > 0: score += 1

        # 4. MOL (EBITDA)
        if dati['MOL_Margin'] is not None and dati['MOL_Margin'] > 0.20: score += 1
        elif dati['MOL_Margin'] is not None and dati['MOL_Margin'] > 0.10: score += 0.5

        # 5. Profit Margin
        if dati['Profit_Margin'] is not None and dati['Profit_Margin'] > 0.08: score += 1

        # 6. ROE
        if dati['ROE'] is not None and dati['ROE'] > risk_free: score += 1

    except Exception as e:
        dati['Note'] += "Errore calcolo score; "

    dati['SCORE'] = score
    dati['Macro_Match'] = check_scenario_match(dati, scenario)
    return dati

# --- 6. ESECUZIONE MASSIVA ---
print(f"\nScaricamento dati storici (Attendere prego...)\n")
try:
    # Scarica i dati storici tutti insieme
    data_history = yf.download(tickers_data, period="1y", interval="1d", progress=True, group_by='ticker')
except Exception as e:
    print(f"Errore download: {e}")
    exit()

results = []
print(f"\nAnalisi puntuale in corso...")

for i, ticker in enumerate(tickers_list):
    try:
        # Pausa tattica per evitare blocchi
        if i % 15 == 0: time.sleep(1)
        if i % 50 == 0: print(f"Processing {i}/{len(tickers_list)}...")

        # Estrazione dati storici
        if len(tickers_list) > 1:
            if ticker not in data_history.columns.levels[0]: continue
            stk_close = data_history[ticker]['Close']
            bench_close = data_history[benchmark_ticker]['Close']
        else:
            stk_close = data_history['Close']
            bench_close = data_history['Close']

        # Calcoli
        rrg = get_rrg_status(stk_close.to_frame(name='Close'), bench_close.to_frame(name='Close'))
        t = yf.Ticker(ticker)
        fund = analizza_fondamentali(t, risk_free_decimal, scenario_type)

        if fund is None: continue

        row = {
            'Ticker': ticker,
            'Settore': fund['Settore'],
            'TARGET MATCH': fund['Macro_Match'],
            'SCORE (Max 6)': fund['SCORE'],
            'RRG Trend': rrg,
            'Prezzo': fund['Prezzo'],
            'P/E': fund['P/E'],
            'D/E Ratio': fund['D/E'],
            'MOL %': fund['MOL_Margin'],
            'ROE %': fund['ROE'],
            'Beta': fund['Beta'],
            'Note': fund['Note']
        }
        results.append(row)
    except Exception as e:
        continue

# --- 7. ESPORTAZIONE EXCEL ---
if not results:
    print("Nessun risultato trovato.")
else:
    df_all = pd.DataFrame(results).sort_values(by='SCORE (Max 6)', ascending=False)
    df_top = df_all[df_all['SCORE (Max 6)'] >= 5]

    def salva_excel_formattato(df, nome_file):
        try:
            writer = pd.ExcelWriter(nome_file, engine='xlsxwriter')
            df.to_excel(writer, sheet_name='Dati', index=False)
            wb = writer.book
            ws = writer.sheets['Dati']

            # Formati
            green = wb.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100'})
            red = wb.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006'})
            perc = wb.add_format({'num_format': '0.00%'})

            ws.set_column('I:J', 10, perc) # Colonne %

            # Formattazione Condizionale
            # RRG
            ws.conditional_format('E2:E500', {'type': 'text', 'criteria': 'containing', 'value': 'LEADING', 'format': green})
            ws.conditional_format('E2:E500', {'type': 'text', 'criteria': 'containing', 'value': 'LAGGING', 'format': red})
            # Target Match
            ws.conditional_format('C2:C500', {'type': 'text', 'criteria': 'begins with', 'value': 'SI', 'format': green})

            writer.close()
            print(f"Salvato: {nome_file}")
        except Exception as e:
            print(f"Errore salvataggio {nome_file}: {e}")

    salva_excel_formattato(df_top, 'Top_Picks.xlsx')
    salva_excel_formattato(df_all, 'Analisi_Completa.xlsx')

    print("\n--- TUTTO COMPLETATO! CONTROLLA LA CARTELLA FILE A SINISTRA ---")

--- SISTEMA DI SELEZIONE AZIONARIA (V6.0 Final) ---

1. Inserisci il Tasso Risk Free USA (es. 4.2): 3.75

2. Trend Bilancio Banche Centrali (Liquidità)?
   [1] In Aumento (Scenario 'Most Liquid')
   [2] In Diminuzione (Scenario 'Least Liquid')
   Digitare 1 o 2: 2

3. Indice di Riferimento (Invio per S&P500): 

SCENARIO RILEVATO: DEFENSIVE (Stock A)

--- AVVIO RECUPERO DATI ---
Tentativo 1: Scaricamento da Database GitHub...
-> Successo! Trovati 503 ticker.

Scaricamento dati storici (Attendere prego...)



/tmp/ipython-input-736263580.py:174: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data_history = yf.download(tickers_data, period="1y", interval="1d", progress=True, group_by='ticker')
[******                13%                       ]  65 of 504 completedERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WBA"}}}
[*********************100%***********************]  504 of 504 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['WBA']: YFPricesMissingError('possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")')



Analisi puntuale in corso...
Processing 0/503...
Processing 50/503...
Processing 100/503...
Processing 150/503...
Processing 200/503...
Processing 250/503...
Processing 300/503...
Processing 350/503...
Processing 400/503...
Processing 450/503...


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WBA"}}}


Processing 500/503...
Salvato: Top_Picks.xlsx
Salvato: Analisi_Completa.xlsx

--- TUTTO COMPLETATO! CONTROLLA LA CARTELLA FILE A SINISTRA ---
